# Day 23 — 시계열 기초: 작은 신호로 배우는 여섯 가지 개념

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-23.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-23.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: 신호 만들기와 시간축

In [ ]:
import numpy as np

fs = 100                       # sampling rate: 초당 100개
duration = 10                  # 초
t = np.arange(0, duration, 1 / fs)

rng = np.random.default_rng(0)             # seed 고정 → 매번 같은 결과
clean = 0.05 * np.sin(2 * np.pi * 5 * t)   # 5 Hz '떨림'
drift = 0.02 * t / duration                # 천천히 기울어지는 baseline
noise = rng.normal(0, 0.02, len(t))        # 잡음
signal = clean + drift + noise

print("표본 개수:", len(signal))
print("표본 사이 간격(초):", 1 / fs)
print("처음 5개 값:", np.round(signal[:5], 3))
print("평균:", round(signal.mean(), 4), " 표준편차:", round(signal.std(), 4))

### 예시 2: 너무 느리게 측정하면 생기는 일 (aliasing)

In [ ]:
import numpy as np

def dominant_freq(x, fs):
    """신호 x에서 가장 강한 주파수(Hz)를 돌려준다."""
    x = x - x.mean()                       # 평균(0 Hz 성분) 제거
    power = np.abs(np.fft.rfft(x)) ** 2
    freqs = np.fft.rfftfreq(len(x), 1 / fs)
    return freqs[np.argmax(power)]

for fs in [100, 20, 8]:
    t = np.arange(0, 10, 1 / fs)
    x = np.sin(2 * np.pi * 5 * t)          # 진짜 주파수는 항상 5 Hz
    print(f"fs={fs:>3} Hz, Nyquist={fs/2:>4} Hz → 측정된 주파수: {dominant_freq(x, fs):.1f} Hz")

### 예시 3: 필터링과 스펙트럼

In [ ]:
import numpy as np
from scipy import signal as sg

fs = 100
t = np.arange(0, 10, 1 / fs)
rng = np.random.default_rng(0)
clean = 0.05 * np.sin(2 * np.pi * 5 * t)
x = clean + 0.02 * t / 10 + rng.normal(0, 0.02, len(t))

# 3–8 Hz만 통과시키는 band-pass filter (Butterworth 4차)
b, a = sg.butter(4, [3, 8], btype="bandpass", fs=fs)
filtered = sg.filtfilt(b, a, x)            # filtfilt: 시간 밀림 없이 필터링

def rmse(u, v):
    return np.sqrt(np.mean((u - v) ** 2))

print("필터 전, 깨끗한 신호와의 차이(RMSE):", round(rmse(x, clean), 4))
print("필터 후, 깨끗한 신호와의 차이(RMSE):", round(rmse(filtered, clean), 4))

# Welch 방법으로 주파수별 세기(power spectrum) 보기
freqs, power = sg.welch(x, fs=fs, nperseg=256)
print("가장 강한 주파수(Hz):", round(freqs[np.argmax(power[1:]) + 1], 2))
print("주파수 해상도(Hz):", round(freqs[1] - freqs[0], 3))

### 예시 4: 결측과 window

In [ ]:
import numpy as np
import pandas as pd

fs = 100
t = np.arange(0, 10, 1 / fs)
rng = np.random.default_rng(0)
x = 0.05 * np.sin(2 * np.pi * 5 * t) + rng.normal(0, 0.02, len(t))

# (1) 결측 만들기: 2.00–2.29초 구간 30개가 빠졌다고 가정
x_missing = x.copy()
x_missing[200:230] = np.nan
s = pd.Series(x_missing, index=t)
print("결측 개수:", s.isna().sum(), "/", len(s))
print("결측 비율(%):", round(100 * s.isna().mean(), 1))

# (2) window: 2초(200개)씩, 겹치지 않게 자르기
win = 2 * fs
rows = []
for start in range(0, len(x_missing) - win + 1, win):
    seg = x_missing[start:start + win]
    rows.append({
        "window_start_s": start / fs,
        "n_missing": int(np.isnan(seg).sum()),
        "std": round(float(np.nanstd(seg)), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

## 직접 해보기

1. `notebooks/` 폴더에 `04_signal_basics.ipynb`를 새로 만듭니다. 첫 셀(Markdown)에 "가상 신호로 시계열 기초 연습. 실제 데이터 아님."이라고 씁니다.
2. **신호 만들기:** 예시 1을 실행합니다. 이어서 아래 셀로 그림을 그립니다.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.plot(t[:300], signal[:300])
plt.xlabel("Time (s)"); plt.ylabel("Acceleration (g, simulated)")
plt.title("First 3 seconds")
plt.show()